In [ ]:
'''
this code was added after the pre print submission in reponse to reviewer request for additional analysis regarding DsbA/D/C 
MauE partner proteins and the quinone usage in bacterial species. This is inconjunction with additional data in the form of 
mutational analysis of conserved lysine and glutamic acids residues and B8FLW7 and H1XRV2, motility assays and experiments in 
a ubiquinone and menaquinone double knockout strain.

the major challenge here is that the uniprot database had many entries culled as part of updates taking place throughout 2025 
2026. The entries culled meant that in order to keep a consistant comparison and not skew results due to false negative incurred
by entire species proteomes being culled, we had to use the march 2025 uniprot backup linked in the github respository
for this publication. This backup is 250M+ protein entries in a single DAT file with the entirity of the associated protein 
metadata entry. Our compute and time resources are limited so to make this feasible we converted the taxid IDs for each species
in our bacterial set into the scientific names creating an associated table added to the Zenodo repository. This allowed us to 
find these species in the backup, we then chunked the backup into chunks of 1M and mapped all the species we had in our dataset
to the respective chunk they have entries in. We then search for each chunk for every entry with a set of IPRs and then kept a 
small set of metadata to add into a new set of data for these IPRs. This was done in parallel using two copies of the same 
notbooks titled 13a and 13b 'gathering up backup data'. This was just a quick and easy way to parallelize the process as the 
backup chunks were distributed on two 2tb harddrives. 

the resulting raw data was then combined in this notebook and used to create binary dataframes of the presence or absence of 
each ipr. The databases are dynamic(they update species names or accessions etc) and our data is static (it represents an api 
pull at a specific time and not a what is necessairly currently in the database) this meant that we would acquire false negatives
from potential mismatches, or species potentially not being in the march 2025 backup. We got around this by repeating the QC
check using the three QC IPRs for the large ribosomal subunit, bacterial RNA polymerase and signal peptidase. That made sure
everything was in accord between our two large analysis and made them comparible. We then combined the two datasets to looks at
bacterial species with MauE VKOR or DsbB and what reducing pathways they had, what partners they might have (DsbA or maybe a 
thioredoxin superfamily member) and if they have IPRs for Ubiquinone or Menaquinone synthesis. 

as noted in the end of the ipynb portions of this notebook out put was used to create bargraphs for supplementary figures.
These numbers are printed out neatly in the later cells. 
'''

In [ ]:
import re
import pandas as pd
import sys
from multiprocessing import Pool, TimeoutError
import time
from pathlib import Path
import csv

In [ ]:
species_df = pandas.read_excel(r"Path\5_Filtered_FBDS_IPRcounts_byspecies_MAUA_names.xlsx")
phylum_df = pandas.read_excel(r"Path\8b_MAUgenes_3cat_3mark_perc_no_can_10more_names.xlsx")

In [ ]:
species_df = pandas.read_excel(r"Path\5_Filtered_FBDS_IPRcounts_byspecies_MAUA_names.xlsx")
phylum_df = pandas.read_excel(r"Path\8b_MAUgenes_3cat_3mark_perc_no_can_10more_names.xlsx")
phyl_list = list(phylum_df['phylum'])
for i in species_df.index:
    phylum = species_df.at[i,'phylum']
    if phylum not in phyl_list:
        species_df = species_df.drop(i)
print(species_df.shape) #should be 20704
species_list = list(species_df['species'])

In [ ]:
species_list = list(species_df['species'])

In [ ]:
'''
this is how i got the species names to search in the uniprot back up chunks
'''
from Bio import Entrez
import time

Entrez.email = "your.email@domain.com"   # required by NCBI


def fetch_taxonomy_names(tax_ids, batch_size=200, sleep_time=0.34):
    """
    Given a list of NCBI taxonomy IDs, return:
        species_names: list aligned to input tax_ids
        missed_species: list of IDs that were not found
        name_lookup: dict {tax_id: scientific_name}
    """

    tax_ids = [str(x).strip() for x in tax_ids]
    unique_ids = list(dict.fromkeys(tax_ids))  # preserve order

    name_lookup = {}

    for start in range(0, len(unique_ids), batch_size):
        batch = unique_ids[start:start + batch_size]

        handle = Entrez.efetch(
            db="taxonomy",
            id=",".join(batch),
            retmode="xml"
        )

        records = Entrez.read(handle)
        handle.close()

        for record in records:
            tax_id = str(record["TaxId"])
            scientific_name = record.get("ScientificName", "None")
            name_lookup[tax_id] = scientific_name

        time.sleep(sleep_time)

    species_names = []
    missed_species = []

    for original_id in tax_ids:
        name = name_lookup.get(original_id)

        if name is None:
            species_names.append("None")
            missed_species.append(original_id)
        else:
            species_names.append(name)

    return species_names, missed_species, name_lookup


# test below 
species_listtest = [212, 2123, 9606]

species_names, missed_species, name_lookup = fetch_taxonomy_names(species_listtest)

print(species_names)
print("Missed:", missed_species)

In [ ]:
species_names, missed_species, name_lookup = fetch_taxonomy_names(species_list)


In [ ]:
taxid2name = pandas.DataFrame()
taxid2name['taxid'] = species_list
taxid2name['species_name'] = species_names

In [ ]:
taxid2name.to_excel(r"Path\taxid2speciesname_table.xlsx")

In [ ]:
#unfortunatly some were missed and gave a na result which means I cant search them in the back up: about 100 species

In [ ]:
'''
here the 250M+ entries were then chunked using 'simple_1M_chunker.py'
the deposited in a seperated folder
'''

In [ ]:
txid_names = pd.read_excel(r"Path\taxid2speciesname_table.xlsx")

In [ ]:
#this mapped every species in the chunks I made

ROOT = Path(r"Path\1M_entry_chunks_260")
OUT = Path(r"Path\os_index.csv")

def extract_unique_os_lines(path: Path):
    seen = set()

    with open(path, "rb", buffering=1024 * 1024 * 8) as f:
        for line in f:
            if line.startswith(b"OS"):
                os_line = line.decode("utf-8", errors="replace").strip()

                if os_line not in seen:
                    seen.add(os_line)
                    yield os_line

with OUT.open("w", newline="", encoding="utf-8") as out:
    writer = csv.writer(out)
    writer.writerow(["os_line", "file"])

    for path in ROOT.glob("*.txt"):
        for os_line in extract_unique_os_lines(path):
            writer.writerow([os_line, str(path)])

print(f"Wrote index to {OUT}")

In [ ]:
import pandas as pd
species_map = pd.read_csv(r"Path\os_index.csv")

In [ ]:
species_names = pd.read_excel(r"Path\taxid2speciesname_table.xlsx")

In [ ]:
names = list(species_names["species_name"].dropna())

In [ ]:
import re
import pandas as pd


#here we search for the species names we get from the table to map them to the chunks
pattern = re.compile("|".join(re.escape(name) for name in names))

dicts = []
count = 0

for os_line, file_path in zip(species_map["os_line"], species_map["file"]):
    match = pattern.search(os_line)
    count += 1

    if match:
        dicts.append({
            "species_name": match.group(0),
            "file_path": file_path
        })

        if len(dicts) % 1000 == 0:
            print(count, len(dicts))

df_mapped = pd.DataFrame.from_records(dicts)
print(df_mapped.shape)

In [ ]:
print(df_mapped)

In [ ]:
df_mapped.to_csv(r"Path\species_indexed2.csv")

In [ ]:
# after this I ran the parallel notebooks to gather the 20k entries I could gather
# now ill import them and merge it with the original df for that ill need taxid2species
full_df1 = pd.read_excel(r"Path\full_df1.xlsx")
full_df2 = pd.read_excel(r"Path\full_df2.xlsx")


In [ ]:
full_df_comb = pd.concat([full_df1,full_df2],ignore_index = True)
full_df_comb = full_df_comb.drop('Unnamed: 0',axis=1)

In [ ]:
print(full_df_comb)

In [ ]:
s2t = pd.read_excel(r"Path\taxid2speciesname_table.xlsx")

In [ ]:
print(full_df_comb.columns)

In [ ]:
for i in full_df_comb.columns:
    print(i,full_df_comb[full_df_comb[i]== True].shape)

In [ ]:
print(s2t.columns)

In [ ]:
tx_full_df = pd.merge(full_df_comb,s2t, left_on='species',right_on='species_name_org',how='left')

In [ ]:
print(tx_full_df)

In [ ]:
dataset_5 = pd.read_excel(r"Path\5_Filtered_FBDS_IPRcounts_byspecies_MAUA_names.xlsx")

In [ ]:
print(dataset_5.columns)

In [ ]:
d5_new = pd.merge(dataset_5,tx_full_df,left_on='species',right_on='taxid')

In [ ]:
print(d5_new)

In [ ]:
cols = list(d5_new.columns)
print(cols)

In [ ]:
'''
these are the columns ill keep. I'll also change them all to either 0 or 1 as a true false alternative to the IPR presence or 
absence. then ill sum get some percentages in another notebook
['species_x', 'IPR009908', 'IPR012932', 'IPR003752', 'IPR047873', 'IPR010243',
'IPR036286', 'IPR026259', 'IPR036560','superkingdom', 'clade', 'phylum', 'class', 'order', 'family', 'genus', 
'scientific_names_phylum','IPR033954', 'IPR036249', 'IPR006370', 'IPR026046', 'dsbD', 'dsbA', 'species_y',
'taxid', 'species_name_org']
these are the columns ill change to 0 or 1:
'IPR009908', 'IPR012932', 'IPR003752', 'IPR047873', 'IPR010243','IPR036286', 'IPR026259', 'IPR036560','IPR033954', 'IPR036249', 'IPR006370', 'IPR026046', 'dsbD', 'dsbA'
'''

In [ ]:
d9 = d5_new[['species_x', 'IPR009908', 'IPR012932', 'IPR003752', 'IPR047873', 'IPR010243',
'IPR036286', 'IPR026259', 'IPR036560','superkingdom', 'clade', 'phylum', 'class', 'order', 'family', 'genus', 
'scientific_names_phylum','IPR033954', 'IPR036249', 'IPR006370', 'IPR026046', 'dsbD', 'dsbA', 'species_y',
'taxid', 'species_name_org']].copy()

In [ ]:
print(d9)

In [ ]:
def make_binary(value):
    if value == True:
        return 1
    elif value == False:
        return 0
    elif value >= 1:
        return 1
    elif value < 1:
        return 0
    elif value == 0:
        return 0
d9_2 = d9.copy() 
print(d9_2)
for i in ['IPR009908', 'IPR012932', 'IPR003752', 'IPR047873', 'IPR010243','IPR036286', 'IPR026259', 'IPR036560','IPR033954', 'IPR036249', 'IPR006370', 'IPR026046', 'dsbD', 'dsbA']:
        d9_2[i] = d9_2[i].map(make_binary)
print(d9_2)

In [ ]:
print(d9_2)

In [ ]:
d9_2.to_excel(r"Path\9_dsbb_maue_vkor_vs_IPRS.xlsx")

In [ ]:
'''
This is where the qc repeat comes in and we lose false negatives. I had to do it in three chunks because our network had a 
shutdown
'''

In [ ]:
fqc1 = pd.read_excel(r"Path\full_df_1_QC.xlsx")
fqc2 = pd.read_excel(r"Path\full_df_QC1.xlsx")
fqc3 = pd.read_excel(r"Path\full_df_QC2.xlsx")

In [ ]:
full_QC = pd.concat([fqc1,fqc2,fqc3],ignore_index = True)


In [ ]:
print(full_QC.columns)

In [ ]:

full_QC = full_QC.drop('Unnamed: 0',axis=1)

In [ ]:

mask = (full_QC['IPR047873'] == True)&(full_QC['IPR010243'] == True)&(full_QC['IPR036286'] == True)
full_QC_checked = full_QC[mask]

In [ ]:
print(full_QC.shape,full_QC_checked.shape)

In [ ]:
print(full_QC,full_QC_checked)

In [ ]:

df_9 = pd.read_excel(r"Path\9_dsbb_maue_vkor_vs_IPRS.xlsx")

In [ ]:

d9b = pd.merge(df_9,full_QC_checked,left_on='species_name_org',right_on='species')

In [ ]:
d9b = d9b.drop('Unnamed: 0',axis=1)

In [ ]:
print(d9b.columns)
print(d9b)

In [ ]:
def make_binary(value):
    if value == True:
        return 1
    elif value == False:
        return 0
    elif value >= 1:
        return 1
    elif value < 1:
        return 0
    elif value == 0:
        return 0
d9b_2 = d9b.copy() 
print(d9b_2)
for i in ['IPR047873_y','IPR010243_y','IPR036286_y']:
        d9b_2[i] = d9b_2[i].map(make_binary)
print(d9b_2)

In [ ]:
#I renamed the new QC columns in excel
d9b_2.to_excel(r"Path\9b_dsbb_maue_vkor_vs_IPRS_QC.xlsx")

In [ ]:
#now that I have all this stuff I need to ad a per phylum breakdown as per lloyd and reveiwer request

df9b= pd.read_excel(r"Path\9b_dsbb_maue_vkor_vs_IPRS_QC.xlsx")
dsbA_or_dsbD = []
for i in df9b.index:
    binary = 0
    if df9b.at[i,'dsbA'] != 0 or df9b.at[i,'dsbD'] != 0:
        binary = 1
        
    dsbA_or_dsbD.append(binary)
df9b['dsbA_or_dsbD'] = dsbA_or_dsbD   
        

In [ ]:
print(df9b['dsbA_or_dsbD'].sum())

In [ ]:
df9c=df9b.copy()
df9c_total = df9b.copy()

mask = (df9c['IPR009908'] == 0) & (df9c['IPR003752'] == 0) & (df9c['IPR012932'] == 0)
df9c = df9c[mask]
df9c = df9c.groupby('phylum').sum()
df9d = df9c[['IPR009908', 'IPR012932', 'IPR003752',
       'IPR047873_QC1', 'IPR010243_QC1', 'IPR036286_QC1', 'IPR026259',
       'IPR036560', 'IPR033954', 'IPR036249',
       'IPR006370', 'IPR026046', 'dsbD', 'dsbA',
       'IPR047873_QC2', 'IPR010243_QC2', 'IPR036286_QC2', 'dsbA_or_dsbD']]

df9c_total = df9c_total.groupby('phylum').sum()
df9d_total = df9c_total[['IPR009908', 'IPR012932', 'IPR003752',
       'IPR047873_QC1', 'IPR010243_QC1', 'IPR036286_QC1', 'IPR026259',
       'IPR036560', 'IPR033954', 'IPR036249',
       'IPR006370', 'IPR026046', 'dsbD', 'dsbA',
       'IPR047873_QC2', 'IPR010243_QC2', 'IPR036286_QC2','dsbA_or_dsbD']]

df9e = pd.merge(df9d,df9d_total,left_on='phylum', right_on='phylum',how='right')
#all the qc checks represent the total because every species has it, so I can use any to calculate a percentage 


In [ ]:
print(df9e.columns)

In [ ]:
df9e = df9e.fillna(0)
print(df9e['IPR047873_QC1_x'],df9e['IPR047873_QC1_y'])

In [ ]:
df9e = df9e.assign(no_path = lambda x: (df9e['IPR047873_QC1_x'] / df9e['IPR047873_QC1_y'] * 100)) #'IPR047873_QC1_x' here is from the none path set and y is the total set
print(df9e)

In [ ]:
df9e = df9e.assign(no_path_with_reducing = lambda x : (df9e['dsbA_or_dsbD_x'] / df9e['IPR047873_QC1_x'] *100 ))#looking only at no paths here


In [ ]:
print(df9e)

In [ ]:
print(df9e.shape)

In [ ]:
names = df9b.groupby(['phylum','scientific_names_phylum']).sum()
listed_names_in_order= [x[1] for x in list(names.index)]
print(listed_names_in_order)

In [ ]:
print(names.index)

In [ ]:
df9e['scientific_names_phylum'] = listed_names_in_order

In [ ]:

df9graph = df9e[['IPR047873_QC1_x','dsbA_or_dsbD_x','IPR047873_QC1_y','no_path', 'no_path_with_reducing','scientific_names_phylum']]
print(df9graph.sort_values('scientific_names_phylum'))

In [ ]:
df9graph.to_excel(r"Path\9c_nopaths_w_and_wo_reducing_and_percentages.xlsx")
df9graph_sorted = df9graph.sort_values('scientific_names_phylum',ascending= False)
print(df9graph_sorted.shape)
#in the excel 9c I rename the _x columns and _y columns manually to make more sense to readers

In [ ]:
'''
now we are going to compare the pathways (dsbb,vkor,maue) and what they have
in each species for species that have each pathway as well as species
that only have one of the three pathways. Well check for DsbA/C/D and trx as 
well as uq and mk markers

first well create single pathway only sets and calculate each percentage
then well do the same without filtering
'''


In [ ]:
df9b = pd.read_excel(r"PATH\9b_dsbb_maue_vkor_vs_IPRS_QC.xlsx")

In [ ]:
#now only filters
M_only = df9b[(df9b['IPR009908'] != 0 ) & (df9b['IPR012932'] == 0) & (df9b['IPR003752'] == 0)]
V_only = df9b[(df9b['IPR009908'] == 0 ) & (df9b['IPR012932'] != 0) & (df9b['IPR003752'] == 0)]
B_only = df9b[(df9b['IPR009908'] == 0 ) & (df9b['IPR012932'] == 0) & (df9b['IPR003752'] != 0)]

In [ ]:
DSB_markers = ['dsbA','dsbD','IPR033954','IPR036249']

M_only_DSB_markers = []
M_only_total_species = M_only['IPR009908'].count()
print('MauE only species',M_only_total_species)
for i in DSB_markers:
    i_count = M_only[i].sum()
    percentage = i_count/M_only_total_species * 100
    M_only_DSB_markers.append(percentage)
    print(i,percentage)
    
B_only_DSB_markers = []
B_only_total_species = B_only['IPR003752'].count()
print('DsbB only species', B_only_total_species)
for i in DSB_markers:
    i_count = B_only[i].sum()
    percentage = i_count/ B_only_total_species * 100
    B_only_DSB_markers.append(percentage)
    print(i,percentage)

V_only_DSB_markers = []
V_only_total_species = V_only['IPR012932'].count()
print('VKOR only species', V_only_total_species)
for i in DSB_markers:
    i_count = V_only[i].sum()
    percentage = i_count/ V_only_total_species * 100
    V_only_DSB_markers.append(percentage)
    print(i,percentage)


In [ ]:
DSB_markers_singles_df = pd.DataFrame()
DSB_markers_singles_df['DSB'] = DSB_markers
DSB_markers_singles_df['DsbB only DSB'] = B_only_DSB_markers
DSB_markers_singles_df['VKOR only DSB'] = V_only_DSB_markers
DSB_markers_singles_df['MauE only DSB'] = M_only_DSB_markers
DSB_markers_singles_df = DSB_markers_singles_df.set_index(DSB_markers_singles_df['DSB'])
DSB_markers_singles_df = DSB_markers_singles_df.drop(columns = ['DSB'])
print(DSB_markers_singles_df)

In [ ]:
#now for mixed pathways
M_mixed = df9b[(df9b['IPR009908'] != 0 )]
V_mixed = df9b[(df9b['IPR012932'] != 0 )]
B_mixed = df9b[(df9b['IPR003752'] != 0 )]

In [ ]:


DSB_markers = ['dsbA','dsbD','IPR033954','IPR036249']

M_mixed_DSB_markers = []
M_mixed_total_species = M_mixed['IPR009908'].count()
print('MauE mixed species',M_mixed_total_species)
for i in DSB_markers:
    i_count = M_mixed[i].sum()
    percentage = i_count / M_mixed_total_species * 100
    M_mixed_DSB_markers.append(percentage)
    print(i,percentage)
    
B_mixed_DSB_markers = []
B_mixed_total_species = B_mixed['IPR003752'].count()
print('DsbB mixed species', B_mixed_total_species)
for i in DSB_markers:
    i_count = B_mixed[i].sum()
    percentage = i_count/ B_mixed_total_species * 100
    B_mixed_DSB_markers.append(percentage)
    print(i,percentage)

V_mixed_DSB_markers = []
V_mixed_total_species = V_mixed['IPR012932'].count()
print('VKOR mixed species', V_mixed_total_species)
for i in DSB_markers:
    i_count = V_mixed[i].sum()
    percentage = i_count/ V_mixed_total_species * 100
    V_mixed_DSB_markers.append(percentage)
    print(i,percentage)

In [ ]:
DSB_markers_mixed_df = pd.DataFrame()
DSB_markers_mixed_df['DSB'] = DSB_markers
DSB_markers_mixed_df['DsbB mixed DSB'] = B_mixed_DSB_markers
DSB_markers_mixed_df['VKOR mixed DSB'] = V_mixed_DSB_markers
DSB_markers_mixed_df['MauE mixed DSB'] = M_mixed_DSB_markers
DSB_markers_mixed_df = DSB_markers_mixed_df.set_index(DSB_markers_mixed_df['DSB'])
DSB_markers_mixed_df = DSB_markers_mixed_df.drop(columns = ['DSB'])
print(DSB_markers_mixed_df)

In [ ]:
DSB_markers_df = pd.concat([DSB_markers_singles_df,DSB_markers_mixed_df],axis = 1)
print(DSB_markers_df)

In [ ]:
DSB_markers_df.to_excel(r"Path\9d_DSB_markersbypathway_species_percentages.xlsx")
#then I manually changed IPR033954 to DsbC and IPR036249 to Trx in the excel

In [ ]:
'''
Okay now I will repeat for quinone markers, I only need to do the dataframe building parts not the filters I can keep those sets
'''

In [ ]:
quinone_markers = ['both','Ubiquinone','Menaquinone','neither']

uq = 'IPR006370'
mq = 'IPR026046'

M_only_Q_markers = []
M_only_total_species = M_only['IPR009908'].count()
print('\n','MauE only species',M_only_total_species,'\n')
for i in quinone_markers:
    if i == 'both':
        both = M_only[(M_only[uq] != 0) & (M_only[mq] != 0)]
        i_count = both['IPR009908'].sum()
    if i == 'Ubiquinone':
        uq_only = M_only[(M_only[uq] != 0) & (M_only[mq] == 0)]
        i_count = uq_only['IPR009908'].sum()
    if i == 'Menaquinone':
        mq_only = M_only[(M_only[uq] == 0) & (M_only[mq] != 0)]
        i_count = mq_only['IPR009908'].sum()
    if i == 'neither':
        neither = M_only[(M_only[uq] == 0) & (M_only[mq] == 0)]
        i_count = neither['IPR009908'].sum()
    
    percentage = i_count/M_only_total_species * 100
    M_only_Q_markers.append(percentage)
    print(i,percentage)

B_only_Q_markers = []
B_only_total_species = B_only['IPR003752'].count()
print('\n','DsbB only species',B_only_total_species,'\n')
for i in quinone_markers:
    if i == 'both':
        both = B_only[(B_only[uq] != 0) & (B_only[mq] != 0)]
        i_count = both['IPR003752'].sum()
    if i == 'Ubiquinone':
        uq_only = B_only[(B_only[uq] != 0) & (B_only[mq] == 0)]
        i_count = uq_only['IPR003752'].sum()
    if i == 'Menaquinone':
        mq_only = B_only[(B_only[uq] == 0) & (B_only[mq] != 0)]
        i_count = mq_only['IPR003752'].sum()
    if i == 'neither':
        neither = B_only[(B_only[uq] == 0) & (B_only[mq] == 0)]
        i_count = neither['IPR003752'].sum()
    
    percentage = i_count/B_only_total_species * 100
    B_only_Q_markers.append(percentage)
    print(i,percentage)
    
V_only_Q_markers = []
V_only_total_species = V_only['IPR012932'].count()
print('\n','VKOR only species',V_only_total_species)
for i in quinone_markers:
    if i == 'both':
        both = V_only[(V_only[uq] != 0) & (V_only[mq] != 0)]
        i_count = both['IPR012932'].sum()
    if i == 'Ubiquinone':
        uq_only = V_only[(V_only[uq] != 0) & (V_only[mq] == 0)]
        i_count = uq_only['IPR012932'].sum()
    if i == 'Menaquinone':
        mq_only = V_only[(V_only[uq] == 0) & (V_only[mq] != 0)]
        i_count = mq_only['IPR012932'].sum()
    if i == 'neither':
        neither = V_only[(V_only[uq] == 0) & (V_only[mq] == 0)]
        i_count = neither['IPR012932'].sum()
    
    percentage = i_count/V_only_total_species * 100
    V_only_Q_markers.append(percentage)
    print(i,percentage)
    
Q_markers_singles_df = pd.DataFrame()
Q_markers_singles_df['Quinone'] = quinone_markers
Q_markers_singles_df['DsbB only Quinone'] = B_only_Q_markers
Q_markers_singles_df['VKOR only Quinone'] = V_only_Q_markers
Q_markers_singles_df['MauE only Quinone'] = M_only_Q_markers
Q_markers_singles_df = Q_markers_singles_df.set_index(Q_markers_singles_df['Quinone'])
Q_markers_singles_df = Q_markers_singles_df.drop(columns = ['Quinone'])
print(Q_markers_singles_df)

In [ ]:
quinone_markers = ['both','Ubiquinone','Menaquinone','neither']

uq = 'IPR006370'
mq = 'IPR026046'

M_mixed_Q_markers = []
M_mixed_total_species = M_mixed['IPR009908'].count()
print('\n','MauE mixed species',M_mixed_total_species,'\n')
for i in quinone_markers:
    if i == 'both':
        both = M_mixed[(M_mixed[uq] != 0) & (M_mixed[mq] != 0)]
        i_count = both['IPR009908'].sum()
    if i == 'Ubiquinone':
        uq_only = M_mixed[(M_mixed[uq] != 0) & (M_mixed[mq] == 0)]
        i_count = uq_only['IPR009908'].sum()
    if i == 'Menaquinone':
        mq_only = M_mixed[(M_mixed[uq] == 0) & (M_mixed[mq] != 0)]
        i_count = mq_only['IPR009908'].sum()
    if i == 'neither':
        neither = M_mixed[(M_mixed[uq] == 0) & (M_mixed[mq] == 0)]
        i_count = neither['IPR009908'].sum()
    
    percentage = i_count/M_mixed_total_species * 100
    M_mixed_Q_markers.append(percentage)
    print(i,percentage)

B_mixed_Q_markers = []
B_mixed_total_species = B_mixed['IPR003752'].count()
print('\n','DsbB mixed species',B_mixed_total_species,'\n')
for i in quinone_markers:
    if i == 'both':
        both = B_mixed[(B_mixed[uq] != 0) & (B_mixed[mq] != 0)]
        i_count = both['IPR003752'].sum()
    if i == 'Ubiquinone':
        uq_only = B_mixed[(B_mixed[uq] != 0) & (B_mixed[mq] == 0)]
        i_count = uq_only['IPR003752'].sum()
    if i == 'Menaquinone':
        mq_only = B_mixed[(B_mixed[uq] == 0) & (B_mixed[mq] != 0)]
        i_count = mq_only['IPR003752'].sum()
    if i == 'neither':
        neither = B_mixed[(B_mixed[uq] == 0) & (B_mixed[mq] == 0)]
        i_count = neither['IPR003752'].sum()
    
    percentage = i_count/B_mixed_total_species * 100
    B_mixed_Q_markers.append(percentage)
    print(i,percentage)
    
V_mixed_Q_markers = []
V_mixed_total_species = V_mixed['IPR012932'].count()
print('\n','VKOR mixed species',V_mixed_total_species)
for i in quinone_markers:
    if i == 'both':
        both = V_mixed[(V_mixed[uq] != 0) & (V_mixed[mq] != 0)]
        i_count = both['IPR012932'].sum()
    if i == 'Ubiquinone':
        uq_only = V_mixed[(V_mixed[uq] != 0) & (V_mixed[mq] == 0)]
        i_count = uq_only['IPR012932'].sum()
    if i == 'Menaquinone':
        mq_only = V_mixed[(V_mixed[uq] == 0) & (V_mixed[mq] != 0)]
        i_count = mq_only['IPR012932'].sum()
    if i == 'neither':
        neither = V_mixed[(V_mixed[uq] == 0) & (V_mixed[mq] == 0)]
        i_count = neither['IPR012932'].sum()
    
    percentage = i_count/V_mixed_total_species * 100
    V_mixed_Q_markers.append(percentage)
    print(i,percentage)
    
Q_markers_mixed_df = pd.DataFrame()
Q_markers_mixed_df['Quinone'] = quinone_markers
Q_markers_mixed_df['DsbB mixed Quinone'] = B_mixed_Q_markers
Q_markers_mixed_df['VKOR mixed Quinone'] = V_mixed_Q_markers
Q_markers_mixed_df['MauE mixed Quinone'] = M_mixed_Q_markers
Q_markers_mixed_df = Q_markers_mixed_df.set_index(Q_markers_mixed_df['Quinone'])
Q_markers_mixed_df = Q_markers_mixed_df.drop(columns = ['Quinone'])
print(Q_markers_mixed_df)

In [ ]:
Q_markers_df = pd.concat([Q_markers_singles_df,Q_markers_mixed_df],axis = 1)
print(Q_markers_df)
Q_markers_df.to_excel(r"Path\9e_Quinone_markersbypathway_species_percentages.xlsx")
#then I manually changed IPR033954 to DsbC and IPR036249 to Trx in the excel

In [ ]:
#counting species with no pathway but have DsbA
df9b = pd.read_excel(r"Path\9b_dsbb_maue_vkor_vs_IPRS_QC.xlsx")

In [ ]:
none = df9b[(df9b['IPR009908'] == 0)&(df9b['IPR003752'] == 0)&(df9b['IPR012932'] == 0)]
print(none['dsbA'].sum())
print(none['IPR047873_QC1'].sum())
print(none.shape)
print(df9b.shape)

In [ ]:
'''
These numbers were used to create some of the supplementary bar graphs. running the ipynb will reconstitute the 
information.
below is some additional work I did to look at the MauE with no Quinones
'''

In [ ]:
'''
The sections below help us get the number of species with MauE that also have MauD

'''

In [ ]:
full_df_QC = pd.read_excel(r"PATH\9b_dsbb_maue_vkor_vs_IPRS_QC.xlsx")
MauD1 = pd.read_excel(r"PATH\full_df_MauD1.xlsx")
MauD2 = pd.read_excel(r"PATH\full_df_MauD2.xlsx")
MauD3 = pd.read_excel(r"PATH\full_df_MauD3.xlsx")
MauD4 = pd.read_excel(r"PATH\full_df_MauD4.xlsx")

full_MD = pd.concat([MauD1,MauD2,MauD3,MauD4],ignore_index = True)
full_MD = full_MD.drop('Unnamed: 0',axis=1)

#mask = (full_MD['IPR047873'] == True)&(full_MD['IPR010243'] == True)&(full_MD['IPR036286'] == True)
#full_MD_checked = full_MD[mask]


In [ ]:
df_9b = pd.read_excel(r"PATH\9b_dsbb_maue_vkor_vs_IPRS_QC.xlsx")
d9f = pd.merge(df_9b,full_MD,left_on='species_name_org',right_on='species')
d9f = d9f.drop('Unnamed: 0',axis=1)

In [ ]:
print(full_MD.columns)

In [ ]:


def make_binary(value):
    if value == True:
        return 1
    elif value == False:
        return 0
    elif value >= 1:
        return 1
    elif value < 1:
        return 0
    elif value == 0:
        return 0

d9f_2 = d9f.copy() 


print(d9f_2)
for i in ['IPR013478']:
        d9f_2[i] = d9f_2[i].map(make_binary)
print(d9f_2)

In [ ]:
d9f_2.to_excel(r"PATH\9f_dsbb_maue_vkor_vs_IPRS_QC_MauD_added.xlsx")

In [ ]:
#now to grab the percentage of MauD in MauE by species 
mauE_cont = d9f_2[(d9f_2['IPR009908']!=0)]
MauE_MauD = mauE_cont[['IPR013478','IPR009908']]
print(mauE_cont.shape)
print(MauE_MauD['IPR013478'].sum())
print(MauE_MauD['IPR009908'].sum())
print('percent species MauE containing MauD:', (MauE_MauD['IPR013478'].sum()/MauE_MauD['IPR009908'].sum())*100, '%')